# Figure 13 -- finite-difference vs autodiff gradients

Loads `results/differentiability/grad_correctness.json`, produced by `bench/differentiability/grad_correctness.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.repo_root() / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("differentiability/grad_correctness.json")
cfg, recs = art["config"], art["data"]["records"]
recs = [r for r in recs if "error" not in r]
if not recs:
    raise SystemExit("grad_correctness.json has no successful rows")

bases = [b for b in cfg["basis"] if any(r["basis"] == b for r in recs)]
wrts = list(cfg["wrt"])

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2)

# Left: the frozen-topology check -- autodiff against a finite difference of the
# SAME function. This is the one that tests autodiff.
# Right: the full-pipeline check, which rebuilds the tree and therefore also
# measures topology sensitivity. Kept apart on purpose.
for ax, field, title in (
    (axes[0], "frozen_rel_l2", "frozen topology (tests autodiff)"),
    (axes[1], "full_pipeline_rel_l2", "full pipeline (rebuilds the tree)"),
):
    for wi, wrt in enumerate(wrts):
        for bi, basis in enumerate(bases):
            sel = sorted(
                (r for r in recs
                 if r["basis"] == basis and r["wrt"] == wrt and r.get(field)),
                key=lambda r: r["theta"],
            )
            if not sel:
                continue
            ax.plot(
                [r["theta"] for r in sel],
                [r[field] for r in sel],
                marker=style.MARKERS[bi % len(style.MARKERS)],
                linestyle=["-", "--"][wi % 2],
                color=style.entity_color(wrt, wi),
                label=f"d/d {wrt} - {basis}",
                markersize=3.6,
                markerfacecolor="white",
                markeredgecolor=style.entity_color(wrt, wi),
            )
    ax.set_yscale("log")
    ax.set_xlabel(r"opening angle $\theta$")
    ax.set_ylabel("relative $L_2$ gradient error")
    ax.set_title(title, fontsize=8)
    style.finish(ax, legend=(ax is axes[0]), legend_kwargs={"loc": "best", "fontsize": 6.2})

fig.tight_layout()
style.footer(
    fig,
    jsonio.config_caption(cfg, ["n", "order", "leaf_size", "precision", "device"])
    + f"\nFD: {cfg['fd_samples']} coords, eps={cfg['fd_eps']:g}",
)
style.save(fig, FIG_DIR / "fig13_grad_correctness.pdf")

for r in recs:
    print(f"{r['basis']:<9s} theta={r['theta']:<4.2f} {r['wrt']:<9s} "
          f"frozen={r['frozen_rel_l2']:.2e} "
          f"full={r.get('full_pipeline_rel_l2')} far_pairs={r['far_pairs']}")


## Caption

Agreement between autodiff and finite-difference gradients of the FMM force
against opening angle $\theta$, differentiating with respect to positions and to
masses. **Left:** both arms perturb the *same* fixed-topology function
(`differentiable_accelerations` on one prepared state), which is what
`differentiable_accelerations` promises to be exact for; this is the test of
autodiff. **Right:** finite differences of the full pipeline, where the tree is
rebuilt at every perturbed point, so this additionally measures sensitivity to a
changed topology. The two are reported separately because merging them would
charge autodiff for topology changes: the acceptance criterion is piecewise
constant in the positions, so a perturbation that moves a pair across the
acceptance boundary changes which interactions exist, and only the finite
difference sees it. Values from
`results/differentiability/grad_correctness.json`.
